# 30 - quality: perplexity and GSM8K (Build Guide Phase 10)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # must run before torch is imported

In [ ]:
!nvidia-smi
!pip uninstall -y -q torchvision torchaudio
!pip install -q torch==2.6.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -q "transformers>=4.45,<5" safetensors fastapi "uvicorn[standard]" httpx openai \
    prometheus-client psutil hypothesis pytest pytest-asyncio pytest-timeout datasets pandas matplotlib

**Run -> Restart session**, then run the first cell again and continue below.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # must run before torch is imported

In [ ]:
!git clone https://github.com/Muskanpandey0304/turingserve.git /kaggle/working/turingserve
%cd /kaggle/working/turingserve
!pip install -q -e . --no-deps
!python scripts/check_versions.py   # expect torch 2.6.0+cu124, Triton 3.2.0, Tesla T4, (7, 5)

In [ ]:
!hf download Qwen/Qwen2.5-1.5B-Instruct --local-dir /kaggle/working/models/Qwen2.5-1.5B-Instruct
!hf download Qwen/Qwen2.5-0.5B-Instruct --local-dir /kaggle/working/models/Qwen2.5-0.5B-Instruct
!hf download anon8231489123/ShareGPT_Vicuna_unfiltered ShareGPT_V3_unfiltered_cleaned_split.json --repo-type dataset --local-dir /kaggle/working/datasets/sharegpt

In [ ]:
os.environ["TS_MODEL_15"] = "/kaggle/working/models/Qwen2.5-1.5B-Instruct"
os.environ["TS_MODEL_05"] = "/kaggle/working/models/Qwen2.5-0.5B-Instruct"
os.environ["TS_SHAREGPT"] = "/kaggle/working/datasets/sharegpt/ShareGPT_V3_unfiltered_cleaned_split.json"
os.environ["TS_KAGGLE"] = "1"

In [ ]:
!python scripts/perplexity.py --model $TS_MODEL_15 --engine hf --dtype float16 --out /kaggle/working/ppl_hf_fp16.json
!python scripts/perplexity.py --model $TS_MODEL_15 --engine turingserve --dtype float16 --out /kaggle/working/ppl_ts_fp16.json
!python scripts/perplexity.py --model $TS_MODEL_15 --engine turingserve --dtype float16 --int8 --out /kaggle/working/ppl_ts_int8.json

In [ ]:
!pip install -q "lm-eval[api]"
!python scripts/check_versions.py   # if torch changed, reinstall the pin and restart the session

In [ ]:
import subprocess, time, requests
log = open("/kaggle/working/server.log", "w")
p = subprocess.Popen(["turingserve", "serve", "--model", os.environ["TS_MODEL_15"], "--served-model-name", "qwen", "--port", "8000"], stdout=log, stderr=subprocess.STDOUT)
for _ in range(240):
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            break
    except Exception:
        pass
    time.sleep(5)

In [ ]:
!lm_eval --model local-chat-completions \
  --model_args model=qwen,base_url=http://localhost:8000/v1/chat/completions,num_concurrent=16,max_retries=3,tokenized_requests=False \
  --tasks gsm8k --num_fewshot 5 --apply_chat_template --fewshot_as_multiturn \
  --gen_kwargs temperature=0 --output_path /kaggle/working/results/gsm8k_fp16 --log_samples

Now INT8. Stop the fp16 server, check the GPU is free, restart with `--quantization int8` and repeat with `--output_path .../gsm8k_int8`. Results (fp16 / int8): perplexity 9.2123 / 9.2270 (+0.16%), GSM8K flexible-extract 0.6050 / 0.6012.

In [ ]:
p.terminate(); p.wait()
time.sleep(5)
!nvidia-smi --query-gpu=memory.used --format=csv


In [ ]:
import subprocess, time, requests
log = open("/kaggle/working/server_int8.log", "w")
p = subprocess.Popen(["turingserve", "serve", "--model", os.environ["TS_MODEL_15"], "--served-model-name", "qwen", "--port", "8000", "--quantization", "int8"], stdout=log, stderr=subprocess.STDOUT)
for _ in range(240):
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            print("server up")
            break
    except Exception:
        pass
    time.sleep(5)


In [ ]:
!lm_eval --model local-chat-completions \
  --model_args model=qwen,base_url=http://localhost:8000/v1/chat/completions,num_concurrent=16,max_retries=3,tokenized_requests=False \
  --tasks gsm8k --num_fewshot 5 --apply_chat_template --fewshot_as_multiturn \
  --gen_kwargs temperature=0 --output_path /kaggle/working/results/gsm8k_int8 --log_samples


In [ ]:
p.terminate(); p.wait()
